# 0. Sample from  all data

In [76]:
import json
import copy
import random

with open('../data/raw/creative_math/creative_math_evaluation_Claude/deepseek-reasoner_Claude_correctness.json', 'r', encoding='utf-8') as f:
    full_data = json.load(f)
    
with open('../data/raw/creative_math/creative_math_evaluation_Claude/qwen-72B-instruct_Claude_correctness.json', 'r', encoding='utf-8') as f:
    full_data.extend(json.load(f))


existing_files = [
    '../data/raw/creative_math/deepseek-reasoner_Claude_correctness_randomly_selected_novelty.json',
    '../data/raw/creative_math/deepseek-reasoner_v2.0_20.json'
]
for file in existing_files:
    with open(file, 'r', encoding='utf-8') as f:
        existing_data = json.load(f)
    existing_ids = set(item['problem_id'] for item in existing_data)
    full_data = [item for item in full_data if item['problem_id'] not in existing_ids]

# subset_ids = set(item['problem_id'] for item in subset_data)
excluded_ids = ['1967_IMO_Problems_6', '2019_USAJMO_Problems_6']
# full_data_ids = set(item['problem_id'] for item in full_data if item['problem_id'] not in excluded_ids)
full_data_ids = set(
    item['problem_id'] 
    for item in full_data 
    if item['problem_id'] not in excluded_ids and item['correctness']['final_decision'] == 'NO' and len(item['cleaned_response'].strip()) > 30
)
len(full_data), len(full_data_ids)

(894, 37)

In [77]:
total_counter = 0
incorrect_counter = 0 
for item in full_data:
    total_counter += 1
    if item['correctness']['final_decision'] == 'NO':
        incorrect_counter += 1
total_counter, incorrect_counter

(894, 45)

In [ ]:
incorrect_items = [
    item for item in full_data if item['correctness']['final_decision'] == 'NO'
]
len_incorrect = len(incorrect_items)
len_incorrect

45

In [79]:
# # incorrect_items[0]
# for item in full_data:
#     if item['problem_id'] == '2009_AMC_12A_Problems_24':
#         print(item['cleaned_response'])
#         print(item['correctness']['final_decision'])

In [80]:
# len(full_data_ids)

In [81]:
# randomly sample 20 ids + 30 ids from full_data_ids
random.seed(42)
sampled_ids = random.sample(list(full_data_ids), 30)
# sampled_ids = random.sample(list(full_data_ids), 30)

# first_batch = sampled_ids[:20]
# second_batch = sampled_ids[20:]

In [102]:
random.seed(42)

sampled_data = []

for _id in sampled_ids:
    sampled_data.append(random.choice([
        item for item in full_data 
        if item['problem_id'] == _id 
        and item['correctness']['final_decision'] == 'NO'
        and len(item['cleaned_response'].strip()) > 30
    ]))

len(sampled_data)

30

In [103]:
with open('../data/raw/creative_math/v3.0.json', 'w', encoding='utf-8') as f:
    json.dump(sampled_data, f, indent=4)

# 1. Collect inference output and export to MTurk format

In [104]:
version = 'v3.0'

In [105]:
import os
import json
import pandas as pd

data_dir = "../data/raw/creative_math/"
json_files = [
    # 'deepseek-reasoner_Claude_correctness_randomly_selected_novelty.json',
    # 'qwen-72B-instruct_Claude_correctness_randomly_selected.json',
    # 'deepseek-reasoner_v2.0_30.json',
    # 'deepseek-reasoner_v2.0_20.json',
    'v3.0.json'
]

all_data = []
for file in json_files:
    model_name = file.split('_')[0]
    with open(os.path.join(data_dir, file), 'r', encoding='utf-8') as f:
        data = json.load(f)
        # If the file contains a list of outputs
        if isinstance(data, list):
            for item in data:
                item['model'] = model_name
                all_data.append(item)
        # If the file contains a dict of outputs
        elif isinstance(data, dict):
            data['model'] = model_name
            all_data.append(data)

df = pd.DataFrame(all_data).drop_duplicates(subset=['problem_id', 'model']).reset_index(drop=True)
df.shape

(30, 18)

In [106]:
df.to_csv('../data/mturk_input/creative_math/' + version + '.csv', index=False)

In [107]:
# df.head()
df.columns

Index(['problem', 'problem_id', 'question_number', 'dataset', 'k', 'n',
       'ground_truth_solutions', 'response', 'raw_cleaned_response',
       'extraction_status', 'cleaned_response', 'all_solutions', 'correctness',
       'reasons', 'solution_provided', 'coarse_grained_novelty',
       'fine_grained_novelty', 'model'],
      dtype='object')

In [108]:
def format_reference_solutions(solutions):
    if isinstance(solutions, list):
        return "\n\n\n==========\n".join([f"Solution {i+1}. {sol}" for i, sol in enumerate(solutions)])
    elif isinstance(solutions, str):
        return solutions
    else:
        return ""
df['cleaned_references'] = df['ground_truth_solutions'].apply(format_reference_solutions)

In [109]:
input_cols = ['problem_id', 'problem', 'cleaned_references', 'cleaned_response', 'model']

output_1 = ['correct', 'incorrect']
output_2 = ['novel', 'not_novel']

In [110]:
# df[input_cols].head(10).to_csv('../data/mturk_input/creative_math_ds.csv', index=False)

- **Clean for math formulas**

In [111]:
import re

def latex_to_parentheses(s):
    # Replace all pairs of $...$ with (...), non-greedy match
    s = re.sub(r'\$(.+?)\$', r'\\(\1\\)', s)
    s = s.replace('\\[', '\\(').replace('\\]', '\\)')
    s = s.replace('==========', '<br><br>==========<br>')
    return s

df['problem'] = df['problem'].apply(latex_to_parentheses)
df['cleaned_response'] = df['cleaned_response'].apply(latex_to_parentheses)
df['cleaned_references'] = df['cleaned_references'].apply(latex_to_parentheses)

- **Clean for mturk upload**

In [112]:
def _safe_replacement(ch: str) -> str:
    """
    Try to map 4-byte mathematical letters to ASCII equivalents.
    If not possible, return empty string (remove).
    """
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return ""  # Unknown Unicode block, drop

    # Example: "MATHEMATICAL BOLD SMALL X"
    if name.startswith("MATHEMATICAL "):
        parts = name.split()
        base = parts[-1]
        if len(base) == 1 and base.isalpha():
            return base.lower() if "SMALL" in name else base.upper()

    # Could extend with more rules later
    return ""


def _clean_string(s: str) -> str:
    """
    Remove/rewrite characters with UTF-8 length > 3 bytes.
    Keeps all standard UTF-8 chars.
    """
    if not isinstance(s, str):
        return s

    cleaned = []
    for ch in s:
        b = ch.encode("utf-8", errors="ignore")
        if len(b) <= 3:
            cleaned.append(ch)
        else:
            repl = _safe_replacement(ch)
            if repl:
                cleaned.append(repl)
            # else: drop character entirely

    return "".join(cleaned)


def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply cleaning to an entire pandas DataFrame.
    Returns a NEW DataFrame with all strings cleaned.
    """
    df_clean = df.copy()

    for col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(_clean_string)

    return df_clean

- **Writing to local** 

In [113]:
import copy

In [114]:
# version = 'v3.0'

In [137]:
df['image_in_problem'] = ''
mturk_input = df[['problem_id', 'problem', 'cleaned_references', 'cleaned_response', 'image_in_problem']]

batch_size = 4
start = 0
mturk_input_data = []
for i in range(0, len(mturk_input), batch_size):
    batch = copy.deepcopy(mturk_input.iloc[i:i+batch_size])
    batch = clean_df(batch)
    mturk_input_data.append(batch)
    batch.to_csv(f'../data/mturk_input/creative_math/{version}/{start+i}-{start+i+batch_size}.csv', index=False, encoding='utf-8')

In [133]:
batch.head()

,problem_id,problem,cleaned_references,cleaned_response,image_in_problem
25,2011_USAJMO_Problems_6,Consider the assertion that for each positive ...,Solution 1. We will show that \(n = 25\) is a ...,We demonstrate that \( n = 175 \) is a counter...,
26,2001_AMC_10_Problems_20,A regular octagon is formed by cutting an isos...,"Solution 1. First, realize that each triangle ...",To find the length of each side of the regular...,
27,2015_USAJMO_Problems_3,Quadrilateral \(APBQ\) is inscribed in circle ...,"Solution 1. [asy] size(8cm); pair A=(1,0); pai...",### Solution 3: Geometric Approach Using Cycli...,
28,2015_USAJMO_Problems_5,Let \(ABCD\) be a cyclic quadrilateral. Prove ...,"Solution 1. Note that lines \(AC, AX\) are iso...",**Forward Direction (⇒):**\n\n1. **Assume Exis...,
29,1990_USAMO_Problems_1,A certain state issues license plates consisti...,Solution 1. Consider license plates of \(n\) d...,To determine the maximum number of distinct li...,


In [117]:
# all_latex_str = ""
# for i, row in df.iterrows():
#     # print(row['problem_id'])
#     # print(row['problem'])
#     # print(row['cleaned_response'])
#     # print(row['cleaned_references'])
#     # print('---')
#     latex_str = f"""\\textbf{{Problem ID}}: {row['problem_id']} \\\\
#     \\textbf{{Problem}}: {row['problem']} \\\\
#     \\textbf{{Model}}: {row['model']} \\\\
#     \\textbf{{Response}}: {row['cleaned_response']} \\\\
#     \\textbf{{Reference Solutions}}: {row['cleaned_references']} \\\\
#     \\vspace{{0.3cm}} \\\\
#     \\hrule \\\\
#     \\vspace{{0.5cm}} \\\\
#     """
#     all_latex_str += latex_str  

In [118]:
# with open('../data/mturk_input/creative_math/v2.0/creative_math_mturk_input.tex', 'w', encoding='utf-8') as f:
#     f.write(all_latex_str)

In [119]:
version = 'v3.0'
df_llm_judges = df[['problem_id', 'correctness', 'coarse_grained_novelty']]
df_llm_judges.to_csv(f'../data/mturk_anno/creative_math/{version}_llmj/llm_judges.csv', index=False)
df_llm_judges.head()

,problem_id,correctness,coarse_grained_novelty
0,2023_AMC_8_Problems_14,"{'claude-3-7-sonnet-20250219': 'NO', 'final_de...",{'final_decision': 'NO'}
1,2018_USAJMO_Problems_2,"{'claude-3-7-sonnet-20250219': 'NO', 'final_de...",{'final_decision': 'NO'}
2,2009_AMC_12A_Problems_24,"{'claude-3-7-sonnet-20250219': 'NO', 'final_de...",{'final_decision': 'NO'}
3,2017_USAMO_Problems_3,"{'claude-3-7-sonnet-20250219': 'NO', 'final_de...",{'final_decision': 'NO'}
4,2008_AIME_I_Problems_10,"{'claude-3-7-sonnet-20250219': 'NO', 'final_de...",{'final_decision': 'NO'}


In [120]:
incorrect_counter = 0
for c in df.correctness.values:
    if eval(str(c))['final_decision'] == 'NO':
        incorrect_counter += 1
incorrect_counter

30

In [123]:
sum(df['cleaned_response'] == '')

0

# 2. (ARCHIVED) Clean math formula for MTurk rendering

In [1]:
mturk_input_data = pd.read_csv('../data/mturk_input/creative_math_all.csv')
mturk_input_data.shape

NameError: name 'pd' is not defined

In [53]:
mturk_input_data.columns

Index(['problem_id', 'problem', 'cleaned_references', 'cleaned_response',
       'model'],
      dtype='object')

In [73]:
print(mturk_input_data['problem'].values[1])

An equilateral triangle is originally painted black.  Each time the triangle is changed, the middle fourth of each black triangle turns white.  After five changes, what fractional part of the original area of the black triangle remains black?
[asy] unitsize(36); fill((0,0)--(2,0)--(1,sqrt(3))--cycle,gray); draw((0,0)--(2,0)--(1,sqrt(3))--cycle,linewidth(1));  fill((4,0)--(6,0)--(5,sqrt(3))--cycle,gray); fill((5,0)--(9/2,sqrt(3)/2)--(11/2,sqrt(3)/2)--cycle,white); draw((5,sqrt(3))--(4,0)--(5,0)--(9/2,sqrt(3)/2)--(11/2,sqrt(3)/2)--(5,0)--(6,0)--cycle,linewidth(1)); fill((8,0)--(10,0)--(9,sqrt(3))--cycle,gray); fill((9,0)--(17/2,sqrt(3)/2)--(19/2,sqrt(3)/2)--cycle,white); fill((17/2,0)--(33/4,sqrt(3)/4)--(35/4,sqrt(3)/4)--cycle,white); fill((9,sqrt(3)/2)--(35/4,3*sqrt(3)/4)--(37/4,3*sqrt(3)/4)--cycle,white); fill((19/2,0)--(37/4,sqrt(3)/4)--(39/4,sqrt(3)/4)--cycle,white); draw((9,sqrt(3))--(35/4,3*sqrt(3)/4)--(37/4,3*sqrt(3)/4)--(9,sqrt(3)/2)--(35/4,3*sqrt(3)/4)--(33/4,sqrt(3)/4)--(35/4,s

In [ ]:
import re

def latex_to_parentheses(s):
    # Replace all pairs of $...$ with (...), non-greedy match
    s = re.sub(r'\$(.+?)\$', r'\\(\1\\)', s)
    s = s.replace('\\[', '\\(').replace('\\]', '\\)')
    s = s.replace('==========', '<br><br>==========<br>')
    return s

mturk_input_data['problem'] = mturk_input_data['problem'].apply(latex_to_parentheses)
mturk_input_data['cleaned_response'] = mturk_input_data['cleaned_response'].apply(latex_to_parentheses)
mturk_input_data['cleaned_references'] = mturk_input_data['cleaned_references'].apply(latex_to_parentheses)


In [ ]:

mturk_input_data['image_in_problem'] = ''
mturk_input_data.loc[1, 'image_in_problem'] = 'https://joeyhou.github.io/CreativityPrism/assets/img/1991_AJHSME_Problems_25.png' 

# def append_image(problem, image_url):
#     if image_url:
#         # return f"{problem}\n\n![Image]({image_url})"
#         problem += '<br><br>'
#         problem += '<img alt="problem image" class="tagimage" src="{image_url}"/>' #f"\n\n![Image]({image_url})"
#         problem += '<br><br>'
#     return problem

# mturk_input_data.loc[1, 'problem'] = append_image(mturk_input_data.loc[1, 'problem'], 'https://joeyhou.github.io/CreativityPrism/assets/img/1991_AJHSME_Problems_25.png' )

In [76]:
mturk_input_data.to_csv('../data/mturk_input/creative_math_all_formatted.csv', index=False)

In [69]:
mturk_input_data.shape

(50, 5)